# Exploratory data analysis

Twitter US Airline Sentiment (Kaggle, `data/raw/Tweets.csv`): class balance, tweet length, frequent words per class, volume over time, and the train/val/test split.

In [ ]:
import re
from collections import Counter

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
from nltk.corpus import stopwords

from sentiment.data import clean, load_raw, split
from sentiment.plots import CLASS_COLORS, CLASS_ORDER

sns.set_theme(style="whitegrid", rc={"grid.color": "#e6e5e0", "axes.edgecolor": "#c3c2b7", "axes.spines.top": False, "axes.spines.right": False})

## Load and clean

In [ ]:
raw = load_raw()
df = clean(raw)
print(f"raw rows: {len(raw):,}  after cleaning: {len(df):,}  dropped: {len(raw) - len(df):,}")
df.head()

## Class balance

Negative tweets dominate, so accuracy alone will flatter a model that leans negative. Later phases report macro precision and F1 too.

In [ ]:
counts = df["label"].value_counts().reindex(CLASS_ORDER)
share = counts / counts.sum()
print(pd.DataFrame({"tweets": counts, "share": share.round(3)}))

fig, ax = plt.subplots(figsize=(6, 3))
ax.barh(CLASS_ORDER, counts, color=[CLASS_COLORS[c] for c in CLASS_ORDER], height=0.6)
for i, (n, s) in enumerate(zip(counts, share)):
    ax.text(n + 100, i, f"{n:,} ({s:.0%})", va="center", color="#52514e")
ax.invert_yaxis()
ax.set_xlim(0, counts.max() * 1.25)
ax.set_xlabel("Tweets")
ax.set_title("Most tweets are negative", loc="left")
plt.show()

## Tweet length

Word counts per class (whitespace split).

In [ ]:
df["words"] = df["text"].str.split().str.len()
print(df.groupby("label")["words"].describe().reindex(CLASS_ORDER).round(1))

fig, ax = plt.subplots(figsize=(6, 3))
sns.boxplot(data=df, x="words", y="label", order=CLASS_ORDER, hue="label", palette=CLASS_COLORS, width=0.5, fliersize=2, ax=ax)
ax.set_xlabel("Words per tweet")
ax.set_ylabel("")
ax.set_title("Negative tweets run longer", loc="left")
plt.show()

## Frequent words per class

Rough tokens only (lowercase words, mentions, URLs and English stopwords removed); real preprocessing comes in Phase 2.

In [ ]:
stop = set(stopwords.words("english"))

def rough_tokens(text):
    text = re.sub(r"@\w+|https?://\S+", " ", text.lower())
    return [w for w in re.findall(r"[a-z']+", text) if w not in stop and len(w) > 1]

top = {label: [w for w, _ in Counter(w for t in group for w in rough_tokens(t)).most_common(15)]
       for label, group in df.groupby("label")["text"]}
pd.DataFrame(top)[CLASS_ORDER]

## Volume over time

The data covers about one week in February 2015 (Pacific time).

In [ ]:
daily = (df.assign(day=df["created_at"].dt.date)
           .groupby(["day", "label"]).size().unstack()[CLASS_ORDER])
print(daily)

fig, ax = plt.subplots(figsize=(7, 3.5))
for c in CLASS_ORDER:
    ax.plot(daily.index, daily[c], color=CLASS_COLORS[c], linewidth=2, marker="o", markersize=4, label=c)
    ax.annotate(c, (daily.index[-1], daily[c].iloc[-1]), xytext=(6, 0), textcoords="offset points", va="center", color="#52514e")
ax.set_ylabel("Tweets per day")
ax.set_title("Daily tweet volume by class", loc="left")
ax.legend(frameon=False, loc="upper left")
fig.autofmt_xdate()
plt.show()

## Train / validation / test split

Stratified 70/15/15 with seed 42; `python -m sentiment.data` writes the same split to `data/processed/`.

In [ ]:
parts = dict(zip(["train", "val", "test"], split(df)))
pd.DataFrame({name: p["label"].value_counts().reindex(CLASS_ORDER) for name, p in parts.items()}).assign(total=lambda t: t.sum(axis=1))